# 50 - TF-IDF para clusterizacao baseline

Este notebook compara representacoes TF-IDF para preparar a primeira clusterizacao da base processada filtrada. A unidade de analise e uma descricao normalizada unica, com `frequencia` preservada para analise e uso posterior como peso.

Esta celula importa as funcoes reutilizaveis do pacote. O notebook fica responsavel pela narrativa da pesquisa, enquanto `datasets.py` cuida da base e `tfidf.py` concentra as configuracoes e execucoes de TF-IDF.

In [1]:
import pandas as pd

from nofis_classifier.datasets import (
    DEFAULT_PERIOD_LABEL,
    FILTERED_MARKET_DATASET_NAME,
    load_processed_items,
    prepare_modeling_items,
    processed_dataset_path,
)
from nofis_classifier.paths import find_project_root
from nofis_classifier.tfidf import (
    BASELINE_TFIDF_CONFIG_NAME,
    default_tfidf_configs,
    run_tfidf_experiments,
    save_tfidf_reports,
)

pd.set_option("display.max_colwidth", 120)

Aqui sao definidos os caminhos do experimento: a raiz do projeto, o Parquet processado que sera usado como entrada e a pasta onde os relatorios de TF-IDF serao gravados.

In [2]:
PROJECT_ROOT = find_project_root()
DATASET_PATH = processed_dataset_path(
    dataset_name=FILTERED_MARKET_DATASET_NAME,
    period_label=DEFAULT_PERIOD_LABEL,
    root=PROJECT_ROOT,
)
REPORT_DIR = PROJECT_ROOT / "reports" / "tfidf"

DATASET_PATH

WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/processed/items-notas-fiscais-filtros-combinados/items-notas-fiscais-filtros-combinados-202501-202504.parquet')

## Carregar e validar base

A base filtrada ja deve estar consolidada por `descricao_normalizada`. A preparacao abaixo valida colunas obrigatorias, rejeita duplicatas e garante que a frequencia total nao foi alterada.

Esta celula transforma a base processada na base de modelagem. Para o TF-IDF, cada `descricao_normalizada` unica vira um documento; a coluna `frequencia` e preservada para analises posteriores, mas nao altera diretamente o ajuste do vetor TF-IDF nesta etapa.

In [3]:
raw_items = load_processed_items(DATASET_PATH)
modeling_items = prepare_modeling_items(raw_items)

assert modeling_items["descricao_normalizada"].is_unique
assert modeling_items["descricao_normalizada"].str.strip().ne("").all()
assert raw_items["frequencia"].sum() == modeling_items["frequencia"].sum()

display(modeling_items.head())
display(
    pd.DataFrame(
        {
            "descricoes_unicas": [len(modeling_items)],
            "frequencia_total": [int(modeling_items["frequencia"].sum())],
            "frequencia_media": [modeling_items["frequencia"].mean()],
            "primeiro_mes": [raw_items["primeiro_mes"].min()],
            "ultimo_mes": [raw_items["ultimo_mes"].max()],
        }
    )
)

,descricao_id,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,0,000 alho branco kg,1,000-ALHO BRANCO KG,1,2025-02,2025-02,1,7032090.0,"Alhos, frescos ou refrigerados, exceto para semeadura"
1,1,000 batata especial kg,1,000-BATATA ESPECIAL KG,1,2025-02,2025-02,1,7019000.0,"Batatas, frescas ou refrigeradas, exceto para semeadura"
2,2,000 cebola conserva kg,1,000-CEBOLA CONSERVA KG,1,2025-02,2025-02,1,7031019.0,"Cebolas, frescas ou refrigeradas, exceto para semeadura"
3,3,000 cenoura kg,1,000-CENOURA KG,1,2025-02,2025-02,1,7061000.0,"Cenouras e nabos, frescos ou refrigerados"
4,4,000 kiwi importado kg,1,000-KIWI IMPORTADO KG,1,2025-02,2025-02,1,8105000.0,"Kiwis (quivis), frescos"


,descricoes_unicas,frequencia_total,frequencia_media,primeiro_mes,ultimo_mes
0,27800,134624,4.84259,2025-01,2025-04


## Comparar configuracoes de TF-IDF

As configuracoes comparam unigramas, unigramas com bigramas e caracteres `char_wb`. A configuracao baseline inicial usa caracteres `(3, 5)` com `min_df=2`, porque descricoes fiscais costumam ter abreviacoes, marcas, erros e variacoes curtas.

Esta e a celula principal da etapa 5 do plano. Ela monta varias configuracoes de TF-IDF e ajusta uma matriz para cada uma, permitindo comparar palavras isoladas, palavras com bigramas, n-gramas de caracteres, `min_df` e `max_features`.

In [4]:
configs = default_tfidf_configs()
summary, vocabulary_samples, vectorizers, matrices = run_tfidf_experiments(modeling_items, configs)

for config in configs:
    matrix = matrices[config.name]
    assert matrix.shape[0] == len(modeling_items)
    assert matrix.shape[1] > 0
    assert matrix.shape[1] <= config.max_features

summary.sort_values(["analyzer", "min_df", "feature_count"])

,name,analyzer,ngram_range,min_df,max_features,max_df,sublinear_tf,norm,document_count,feature_count,non_zero_values,total_cells,density,sparsity
2,char_wb_3_5_min_df_2,char_wb,"(3, 5)",2,30000,0.95,True,l2,27800,30000,1615099,834000000,0.001937,0.998063
5,char_wb_3_5_min_df_5,char_wb,"(3, 5)",5,30000,0.95,True,l2,27800,23507,1592518,653494600,0.002437,0.997563
0,word_unigram_min_df_2,word,"(1, 1)",2,10000,0.95,True,l2,27800,6676,128897,185592800,0.000695,0.999305
1,word_unigram_bigram_min_df_2,word,"(1, 2)",2,10000,0.95,True,l2,27800,10000,173830,278000000,0.000625,0.999375
3,word_unigram_min_df_5,word,"(1, 1)",5,10000,0.95,True,l2,27800,3231,119866,89821800,0.001334,0.998666
4,word_unigram_bigram_min_df_5,word,"(1, 2)",5,10000,0.95,True,l2,27800,6469,161352,179838200,0.000897,0.999103


Depois da comparacao, esta celula separa a configuracao baseline. A baseline atual usa `char_wb`, `ngram_range=(3, 5)`, `min_df=2` e `max_features=30000`, uma escolha adequada para descricoes fiscais com abreviacoes, marcas, erros de digitacao e variacoes curtas.

In [5]:
baseline_vectorizer = vectorizers[BASELINE_TFIDF_CONFIG_NAME]
baseline_matrix = matrices[BASELINE_TFIDF_CONFIG_NAME]

display(summary[summary["name"] == BASELINE_TFIDF_CONFIG_NAME])
display(vocabulary_samples[vocabulary_samples["config_name"] == BASELINE_TFIDF_CONFIG_NAME].head(40))

baseline_matrix.shape

,name,analyzer,ngram_range,min_df,max_features,max_df,sublinear_tf,norm,document_count,feature_count,non_zero_values,total_cells,density,sparsity
2,char_wb_3_5_min_df_2,char_wb,"(3, 5)",2,30000,0.95,True,l2,27800,30000,1615099,834000000,0.001937,0.998063


,config_name,sample_type,description,frequency,rank,term,value
115,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,1,0,NaN
116,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,2,00,NaN
117,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,3,00,NaN
118,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,4,000,NaN
119,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,5,000,NaN
120,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,6,0000,NaN
121,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,7,0001,NaN
122,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,8,0002,NaN
123,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,9,0005,NaN
124,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,10,000x,NaN


(27800, 30000)

## Inspecionar documentos vetorizados

Esta etapa permite escolher uma configuracao de TF-IDF e verificar, documento a documento, quais termos receberam maior peso. A ideia e aproximar a matriz esparsa das descricoes legiveis para avaliar se a representacao esta capturando sinais uteis ou apenas fragmentos ruidosos.

In [21]:
BASELINE_TFIDF_CONFIG_NAME = "word_unigram_bigram_min_df_5"

TFIDF_CONFIG_TO_INSPECT = BASELINE_TFIDF_CONFIG_NAME
DOCUMENTS_TO_INSPECT = 12
TERMS_PER_DOCUMENT = 10
SAMPLE_BY = "aleatorio"  # opcoes: "frequencia", "aleatorio", "inicio"
RANDOM_STATE = 42

if TFIDF_CONFIG_TO_INSPECT not in vectorizers:
    raise ValueError(
        f"Configuracao desconhecida: {TFIDF_CONFIG_TO_INSPECT}. "
        f"Opcoes disponiveis: {sorted(vectorizers)}"
    )

inspection_vectorizer = vectorizers[TFIDF_CONFIG_TO_INSPECT]
inspection_matrix = matrices[TFIDF_CONFIG_TO_INSPECT]
inspection_feature_names = inspection_vectorizer.get_feature_names_out()

if SAMPLE_BY == "frequencia":
    inspection_rows = modeling_items.nlargest(DOCUMENTS_TO_INSPECT, "frequencia")
elif SAMPLE_BY == "aleatorio":
    inspection_rows = modeling_items.sample(
        min(DOCUMENTS_TO_INSPECT, len(modeling_items)),
        random_state=RANDOM_STATE,
    )
elif SAMPLE_BY == "inicio":
    inspection_rows = modeling_items.head(DOCUMENTS_TO_INSPECT)
else:
    raise ValueError('SAMPLE_BY deve ser "frequencia", "aleatorio" ou "inicio"')

inspection_records = []
for _, row in inspection_rows.iterrows():
    row_vector = inspection_matrix.getrow(int(row["descricao_id"]))
    top_terms = []

    if row_vector.nnz:
        local_order = row_vector.data.argsort()[::-1][:TERMS_PER_DOCUMENT]
        top_terms = [
            f"{inspection_feature_names[row_vector.indices[position]]} ({row_vector.data[position]:.3f})"
            for position in local_order
        ]

    inspection_records.append(
        {
            "config_name": TFIDF_CONFIG_TO_INSPECT,
            "descricao_id": int(row["descricao_id"]),
            "descricao_normalizada": row["descricao_normalizada"],
            "frequencia": int(row["frequencia"]),
            "termos_tfidf_mais_fortes": ", ".join(top_terms),
            "termos_com_peso": int(row_vector.nnz),
        }
    )

display(summary[summary["name"] == TFIDF_CONFIG_TO_INSPECT])
pd.DataFrame.from_records(inspection_records)


,name,analyzer,ngram_range,min_df,max_features,max_df,sublinear_tf,norm,document_count,feature_count,non_zero_values,total_cells,density,sparsity
4,word_unigram_bigram_min_df_5,word,"(1, 2)",5,10000,0.95,True,l2,27800,6469,161352,179838200,0.000897,0.999103


,config_name,descricao_id,descricao_normalizada,frequencia,termos_tfidf_mais_fortes,termos_com_peso
0,word_unigram_bigram_min_df_5,22811,racumin po 1 kg,1,"po kg (0.815), po (0.456), kg (0.358)",3
1,word_unigram_bigram_min_df_5,6032,ceia,87,ceia (1.000),1
2,word_unigram_bigram_min_df_5,23468,relogio de parede,2,"de parede (0.692), parede (0.685), de (0.229)",3
3,word_unigram_bigram_min_df_5,10884,file de peito de frango rico,4,"rico (0.446), de peito (0.371), peito de (0.361), file de (0.333), peito (0.313), de frango (0.311), file (0.297), f...",9
4,word_unigram_bigram_min_df_5,3711,berinjela cx 13,4,"berinjela (0.648), 13 (0.630), cx (0.428)",3
5,word_unigram_bigram_min_df_5,15869,limpa radiador 200ml 246 orbi gtin 7898314110132,2,"limpa radiador (0.490), radiador (0.462), orbi (0.446), gtin (0.394), limpa (0.318), 200ml (0.301)",6
6,word_unigram_bigram_min_df_5,11177,flv manga kg,4,"manga kg (0.673), manga (0.490), flv (0.488), kg (0.261)",4
7,word_unigram_bigram_min_df_5,22017,presunto de pernil,102,"de pernil (0.544), presunto de (0.529), pernil (0.458), presunto (0.423), de (0.188)",5
8,word_unigram_bigram_min_df_5,19138,ovo branco cuba f 27 02 v 26 03,1,"26 (0.429), 03 (0.414), ovo branco (0.406), 27 (0.385), 02 (0.376), ovo (0.332), branco (0.282)",7
9,word_unigram_bigram_min_df_5,15227,leite em po betania 200g,1,"betania 200g (0.525), betania (0.423), leite em (0.409), 200g (0.308), em po (0.299), po (0.259), leite (0.257), em ...",8


## Salvar relatorios e registro de experimentos

Os arquivos em `reports/tfidf/` registram a comparacao das representacoes, amostras do vocabulario e um registro append-only dos experimentos.

Por fim, esta celula salva a evidencia do experimento. O resumo mostra o tamanho e a esparsidade de cada matriz, as amostras ajudam a inspecionar o vocabulario gerado e o registro append-only preserva metadados para reproducibilidade no TCC.

In [10]:
registry = save_tfidf_reports(
    summary,
    vocabulary_samples,
    REPORT_DIR,
    dataset_name=FILTERED_MARKET_DATASET_NAME,
    period_label=DEFAULT_PERIOD_LABEL,
)

display(registry.tail())
sorted(path.relative_to(PROJECT_ROOT) for path in REPORT_DIR.glob("*.csv"))

,experiment_id,created_at,stage,dataset_name,period_label,document_count,config_name,parameters_json,metrics_json,artifacts_json,notes
19,tfidf-items-notas-fiscais-filtros-combinados-202501-202504-word_unigram_min_df_2-20260907T215937Z,2026-09-07T21:59:37Z,tfidf,items-notas-fiscais-filtros-combinados,202501-202504,27800,word_unigram_min_df_2,"{""analyzer"": ""word"", ""max_df"": 0.95, ""max_features"": 10000, ""min_df"": 2, ""ngram_range"": ""(1, 1)"", ""norm"": ""l2"", ""sub...","{""density"": 0.0006945150889474161, ""feature_count"": 6676, ""non_zero_values"": 128897, ""sparsity"": 0.9993054849110525}","{""summary_csv"": ""C:\\Users\\samue\\Scripts\\Nofis-Classifier\\reports\\tfidf\\tfidf_experiments_summary.csv"", ""vocab...",First TF-IDF comparison for filtered market dataset.
20,tfidf-items-notas-fiscais-filtros-combinados-202501-202504-word_unigram_min_df_5-20260907T203705Z,2026-09-07T20:37:05Z,tfidf,items-notas-fiscais-filtros-combinados,202501-202504,27800,word_unigram_min_df_5,"{""analyzer"": ""word"", ""max_df"": 0.95, ""max_features"": 10000, ""min_df"": 5, ""ngram_range"": ""(1, 1)"", ""norm"": ""l2"", ""sub...","{""density"": 0.0013344867281662135, ""feature_count"": 3231, ""non_zero_values"": 119866, ""sparsity"": 0.9986655132718338}","{""summary_csv"": ""C:\\Users\\samue\\Scripts\\Nofis-Classifier\\reports\\tfidf\\tfidf_experiments_summary.csv"", ""vocab...",First TF-IDF comparison for filtered market dataset.
21,tfidf-items-notas-fiscais-filtros-combinados-202501-202504-word_unigram_min_df_5-20260907T205516Z,2026-09-07T20:55:16Z,tfidf,items-notas-fiscais-filtros-combinados,202501-202504,27800,word_unigram_min_df_5,"{""analyzer"": ""word"", ""max_df"": 0.95, ""max_features"": 10000, ""min_df"": 5, ""ngram_range"": ""(1, 1)"", ""norm"": ""l2"", ""sub...","{""density"": 0.0013344867281662135, ""feature_count"": 3231, ""non_zero_values"": 119866, ""sparsity"": 0.9986655132718338}","{""summary_csv"": ""C:\\Users\\samue\\Scripts\\Nofis-Classifier\\reports\\tfidf\\tfidf_experiments_summary.csv"", ""vocab...",First TF-IDF comparison for filtered market dataset.
22,tfidf-items-notas-fiscais-filtros-combinados-202501-202504-word_unigram_min_df_5-20260907T215357Z,2026-09-07T21:53:57Z,tfidf,items-notas-fiscais-filtros-combinados,202501-202504,27800,word_unigram_min_df_5,"{""analyzer"": ""word"", ""max_df"": 0.95, ""max_features"": 10000, ""min_df"": 5, ""ngram_range"": ""(1, 1)"", ""norm"": ""l2"", ""sub...","{""density"": 0.0013344867281662135, ""feature_count"": 3231, ""non_zero_values"": 119866, ""sparsity"": 0.9986655132718338}","{""summary_csv"": ""C:\\Users\\samue\\Scripts\\Nofis-Classifier\\reports\\tfidf\\tfidf_experiments_summary.csv"", ""vocab...",First TF-IDF comparison for filtered market dataset.
23,tfidf-items-notas-fiscais-filtros-combinados-202501-202504-word_unigram_min_df_5-20260907T215937Z,2026-09-07T21:59:37Z,tfidf,items-notas-fiscais-filtros-combinados,202501-202504,27800,word_unigram_min_df_5,"{""analyzer"": ""word"", ""max_df"": 0.95, ""max_features"": 10000, ""min_df"": 5, ""ngram_range"": ""(1, 1)"", ""norm"": ""l2"", ""sub...","{""density"": 0.0013344867281662135, ""feature_count"": 3231, ""non_zero_values"": 119866, ""sparsity"": 0.9986655132718338}","{""summary_csv"": ""C:\\Users\\samue\\Scripts\\Nofis-Classifier\\reports\\tfidf\\tfidf_experiments_summary.csv"", ""vocab...",First TF-IDF comparison for filtered market dataset.


[WindowsPath('reports/tfidf/experiment_registry.csv'),
 WindowsPath('reports/tfidf/tfidf_experiments_summary.csv'),
 WindowsPath('reports/tfidf/tfidf_vocabulary_samples.csv')]